<a href="https://colab.research.google.com/github/soralh1611/vertex-ai/blob/main/AI_Powered_Loan_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.cloud import aiplatform

# Define your project variables
PROJECT_ID = "soral-vertex-a"
REGION = "us-central1"
BUCKET_URI = "gs://soral-lms_bucket" # Used for storing model artifacts

# Initialize the Vertex AI SDK
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

print(f"Vertex AI SDK initialized for project {PROJECT_ID}")

Vertex AI SDK initialized for project soral-vertex-a


In [3]:
from google.colab import auth
auth.authenticate_user()

import vertexai
vertexai.init(project="soral-vertex-a", location="us-central1")

In [5]:
from google.cloud import storage

def create_new_bucket(bucket_name, location="us-central1"):
    """Creates a new bucket in the specified location."""
    storage_client = storage.Client(project=PROJECT_ID)

    # 1. Clean name (remove gs:// if accidentally added)
    clean_name = bucket_name.replace("gs://", "").lower()

    try:
        # 2. Check if it already exists
        if storage_client.lookup_bucket(clean_name):
            print(f"⚠️ Bucket '{clean_name}' already exists.")
            return storage_client.get_bucket(clean_name)

        # 3. Create the bucket
        bucket = storage_client.create_bucket(clean_name, location=location)

        # 4. Optional: Enable Uniform Bucket-Level Access (Recommended for AI projects)
        bucket.iam_configuration.uniform_bucket_level_access_enabled = True
        bucket.patch()

        print(f"✅ Success: Bucket '{bucket.name}' created in {location}")
        return bucket

    except Exception as e:
        print(f"❌ Error creating bucket: {e}")

# Call the function with a unique name
# Tip: Use your name or project ID as a prefix
NEW_BUCKET_NAME = "lms-reports-soral-2025"
my_bucket = create_new_bucket(NEW_BUCKET_NAME)

✅ Success: Bucket 'lms-reports-soral-2025' created in us-central1


In [26]:
pip install faker reportlab

In [7]:
from faker import Faker
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import random

fake = Faker()

def generate_bank_statement(filename, account_holder):
    doc = SimpleDocTemplate(filename)
    elements = []
    styles = getSampleStyleSheet()

    # Header
    elements.append(Paragraph(f"<b>Bank of Vertex AI - Monthly Statement</b>", styles['Title']))
    elements.append(Paragraph(f"Account Holder: {account_holder}", styles['Normal']))
    elements.append(Paragraph(f"Statement Period: Dec 2025", styles['Normal']))

    # Transaction Data
    data = [["Date", "Description", "Amount", "Balance"]]
    balance = 5000.00
    for _ in range(15):
        date = f"2025-12-{random.randint(1, 20):02d}"
        desc = fake.company()
        amt = round(random.uniform(-500, 1000), 2)
        balance += amt
        data.append([date, desc, f"${amt}", f"${round(balance, 2)}"])

    # Table Styling
    t = Table(data)
    t.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('GRID', (0, 0), (-1, -1), 1, colors.black)
    ]))
    elements.append(t)
    doc.build(elements)

generate_bank_statement("bank_statement_demo.pdf", "John Doe")

In [8]:
pip install faker faker-credit-score reportlab

In [19]:
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from faker_credit_score import CreditScore
from faker.providers import DynamicProvider
from faker.providers import BaseProvider

fake = Faker()
fake.add_provider(CreditScore)

def generate_credit_report(filename, applicant_name):
    doc = SimpleDocTemplate(filename, pagesize=LETTER)
    styles = getSampleStyleSheet()
    elements = []

    # Custom Style for "Confidential" Header
    header_style = ParagraphStyle('HeaderStyle', parent=styles['Normal'], fontSize=10, textColor=colors.red)

    # 1. Header Section
    elements.append(Paragraph("EQUIFAX - CONFIDENTIAL CONSUMER CREDIT FILE", header_style))
    elements.append(Spacer(1, 12))
    elements.append(Paragraph(f"<b>Subject:</b> {applicant_name}", styles['Title']))
    elements.append(Paragraph(f"<b>File Number:</b> {fake.uuid4()}", styles['Normal']))
    elements.append(Paragraph(f"<b>Date of Report:</b> Dec 21, 2025", styles['Normal']))
    elements.append(Spacer(1, 20))

    # 2. Credit Score Section (The "Big Number")
    score = fake.credit_score()
    score_data = [[f"EQUIFAX BEACON 5.0 SCORE: {score}"]]
    score_table = Table(score_data, colWidths=[400])
    score_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.lightgrey),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTSIZE', (0, 0), (-1, -1), 18),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 10),
    ]))
    elements.append(score_table)
    elements.append(Spacer(1, 20))

    # 3. Trade Lines (Credit Accounts)
    elements.append(Paragraph("<b>ACCOUNT HISTORY (TRADE LINES)</b>", styles['Heading2']))
    trade_data = [["Creditor", "Account Type", "Balance", "Status"]]

    # 1. DEFINE the class first
    class BankProvider(BaseProvider):
      def bank_name(self):
          banks = [
              "Chase Bank", "Wells Fargo", "Bank of America",
              "Vertex AI Financial", "Gemini Trust", "Goldman Sachs",
              "PNC Bank", "Citigroup", "Barclays"
          ]
          return self.random_element(banks)
    # 4. Add your custom provider to the Faker instance
    fake.add_provider(BankProvider)
    for _ in range(5):
        trade_data.append([
            fake.bank_name(),
            random.choice(["Revolving", "Installment", "Mortgage"]),
            f"${fake.random_int(0, 15000)}",
            random.choice(["Current", "30 Days Past Due", "Paid as Agreed"])
        ])

    t = Table(trade_data, colWidths=[150, 100, 80, 120])
    t.setStyle(TableStyle([
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('BACKGROUND', (0, 0), (-1, 0), colors.whitesmoke)
    ]))
    elements.append(t)

    doc.build(elements)

    from google.cloud import storage


def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    storage_client = storage.Client()
    clean_name = bucket_name.replace("gs://", "")
    bucket = storage_client.get_bucket(clean_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"✅ Success: Uploaded {source_file_name} to {clean_name}")

# --- EXECUTION STEPS ---

# Set your names
MY_BUCKET = "lms-reports-soral-2025"
FILE_NAME = "synthetic_report.pdf"

# STEP 1: Generate the file (Fixes Errno 2)
generate_credit_report(FILE_NAME, "Alex Rivera")

# STEP 2: Now that the file exists, upload it (Fixes 404)
upload_to_gcs(MY_BUCKET, FILE_NAME, "reports/december_report_01.pdf")



✅ Success: Uploaded synthetic_report.pdf to lms-reports-soral-2025


In [22]:
import os
import random
from faker import Faker
from faker.providers import BaseProvider
from faker_credit_score import CreditScore
from google.cloud import storage
from google.cloud.storage import transfer_manager
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

# 1. SETUP
fake = Faker()
class BankProvider(BaseProvider):
    def bank_name(self):
        return self.random_element(["Chase", "Wells Fargo", "Vertex AI Bank", "Gemini Trust"])

fake.add_provider(BankProvider)
fake.add_provider(CreditScore)

BUCKET_NAME = "lms-reports-soral-2025"
LOCAL_DIR = "bulk_data_reports"
os.makedirs(LOCAL_DIR, exist_ok=True)
styles = getSampleStyleSheet()

# 2. DATA GENERATION FUNCTION
def generate_full_report(i):
    name = fake.name()
    u_id = f"{i:04d}"
    filename = os.path.join(LOCAL_DIR, f"report_{u_id}.pdf")

    doc = SimpleDocTemplate(filename)

    # --- CRITICAL: Create a NEW story list for every file ---
    story = []

    # Add Title
    story.append(Paragraph(f"<b>Financial Audit: {name}</b>", styles['Title']))
    story.append(Spacer(1, 12))

    # Add Financial Summary
    summary_data = [
        ["Metric", "Value"],
        ["Credit Score", str(fake.credit_score())],
        ["Monthly Income", f"${random.randint(3000, 12000)}"],
        ["Primary Bank", fake.bank_name()]
    ]
    summary_table = Table(summary_data, colWidths=[150, 150])
    summary_table.setStyle(TableStyle([('BACKGROUND', (0,0), (-1,0), colors.lightgrey), ('GRID', (0,0), (-1,-1), 1, colors.black)]))
    story.append(summary_table)
    story.append(Spacer(1, 20))

    # Add 15 Mock Transactions (Ensures file is NOT empty)
    trans_data = [["Date", "Merchant", "Amount", "Type"]]
    for _ in range(15):
        trans_data.append([
            str(fake.date_this_year()),
            fake.company(),
            f"${random.randint(-1000, 2000)}",
            random.choice(["Debit", "Credit", "ACH"])
        ])

    trans_table = Table(trans_data, colWidths=[80, 150, 80, 80])
    trans_table.setStyle(TableStyle([('GRID', (0,0), (-1,-1), 0.5, colors.grey), ('FONTSIZE', (0,0), (-1,-1), 8)]))
    story.append(trans_table)

    # FINAL STEP: Build PDF
    doc.build(story)
    return filename

# 3. RUN & UPLOAD
def run_bulk_and_upload(count=1000):
    all_filenames = []
    print(f"🛠️ Generating {count} data-rich reports...")
    for i in range(count):
        all_filenames.append(os.path.basename(generate_full_report(i)))
        if i % 100 == 0: print(f"Progress: {i}/{count}")

    print("🚀 Bulk Uploading to GCS...")
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)

    transfer_manager.upload_many_from_filenames(
        bucket,
        all_filenames,
        source_directory=LOCAL_DIR,
        max_workers=8
    )
    print("✅ All 1,000 files uploaded with data.")

run_bulk_and_upload(1000)

🛠️ Generating 1000 data-rich reports...
Progress: 0/1000
Progress: 100/1000
Progress: 200/1000
Progress: 300/1000
Progress: 400/1000
Progress: 500/1000
Progress: 600/1000
Progress: 700/1000
Progress: 800/1000
Progress: 900/1000
🚀 Bulk Uploading to GCS...
✅ All 1,000 files uploaded with data.


In [27]:
pip install streamlit google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 41.4 MB/s eta 0:00:00


In [33]:
!pip install -q streamlit
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [60]:
# Install the latest Gen AI SDK (Gemini 3 requires version 1.51.0+)
!pip install -q -U google-genai streamlit

# Authenticate Colab to your Google Cloud Account
from google.colab import auth
auth.authenticate_user()

In [65]:
import vertexai
from vertexai.generative_models import GenerativeModel

# Double-check the project ID is exactly 'soral-vertex-a'
PROJECT_ID = "soral-vertex-a"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

try:
    # Use the base name; Vertex AI handles the versioning
    model = GenerativeModel("gemini-2.5-flash")
    response = model.generate_content("Is the API working?")
    print("Success:", response.text)
except Exception as e:
    print("Error still persists:", e)

Success: I need more context to answer that!

*   **Which API are you referring to?** (e.g., Google Maps API, a specific company's internal API, an API for a particular service like Twitter or OpenAI, etc.)
*   **Are you experiencing a specific error or issue?**
*   **What are you trying to do with the API?**

As an AI, I don't have direct access to check the operational status of arbitrary external APIs.

However, if you tell me which API you're asking about, I can give you advice on how to check its status, such as:

1.  **Checking its official status page:** Many major APIs (like AWS, Google Cloud, Stripe, Twilio) have status pages you can visit.
2.  **Looking at recent announcements or forums:** Developers often report outages there.
3.  **Trying to make a test request yourself:** Using tools like cURL, Postman, or a simple script in your code.


In [84]:
%%writefile app.py
import streamlit as st
from google import genai
from google.genai import types

# --- 1. CONFIGURATION ---
PROJECT_ID = "soral-vertex-a"
LOCATION = "us-central1"
# Change this to a verified stable model ID
MODEL_ID = "gemini-2.5-flash"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

st.set_page_config(page_title="LMS Agent", layout="centered")
st.title("🏦 AI Loan Officer")

# --- 2. SESSION STATE ---
if "messages" not in st.session_state:
    st.session_state.messages = []
if "step" not in st.session_state:
    st.session_state.step = "CHAT"

# --- 3. SYSTEM INSTRUCTIONS ---
SYSTEM_PROMPT = """
You are a professional loan officer.
Phase 1: Chat with the user to get their Name, Monthly Income, and Credit Score.
Phase 2: If Income > $3,000 and Score > 650, tell them they are 'Pre-Qualified' and ask them to upload their Bureau PDF.
Phase 3: If they don't meet criteria, politely decline.
"""

# --- 4. DISPLAY CHAT ---
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --- 5. DYNAMIC UI: UPLOADER ---
if st.session_state.step == "UPLOAD":
    st.info("Please upload your Credit Bureau Report to finalize verification.")
    uploaded_file = st.file_uploader("Upload PDF", type="pdf")
    if uploaded_file:
        st.success("File received! Analyzing...")
        # Add logic to send PDF to Gemini here
        st.session_state.step = "ANALYZING"

# --- 6. CHAT INPUT ---
if prompt := st.chat_input("Tell me about your loan needs..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Call Gemini
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.7
        )
    )

    # Display and Save Agent Response
    with st.chat_message("assistant"):
        st.markdown(response.text)
        # Check if the AI wants to move to upload
        if "upload" in response.text.lower() or "pre-qualified" in response.text.lower():
            st.session_state.step = "UPLOAD"
            st.rerun()

    st.session_state.messages.append({"role": "assistant", "content": response.text})

Overwriting app.py


In [85]:
import urllib
print("Your Tunnel Password (IP) is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

Your Tunnel Password (IP) is: 35.243.212.97


In [86]:
# Install localtunnel globally
!npm install -g localtunnel

# Run the app and tunnel to port 8501
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
changed 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://two-chefs-rush.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.243.212.97:8501

  Stopping...
^C


In [87]:
!pip install -q gradio google-genai

In [117]:
import gradio as gr
from google import genai

# --- CONFIG ---
PROJECT_ID = "soral-vertex-a"
LOCATION = "us-central1"
MODEL_ID = "gemini-2.0-flash-001" # Your Gemini 2 family model

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

def loan_agent_chat(message, history):
    # message: the current user input
    # history: previous chat turns

    system_instruction = """
        ROLE: You are 'Artha', the friendly, high-energy Loan Concierge. Also, mention your name just once unless asked again.
        TONE: Enthusiastic, clear, and professional. Use light humor and emojis to make the user smile, but stay laser-focused on safety.

        WORKFLOW:
        1. GREET: Start with a warm, playful welcome.
        2. INTAKE: To check credit, you MUST gather: Full Name, Monthly Income, and a Government ID Number.
        3. PLAYFUL DATA REQUESTS:
          - Instead of 'Enter ID', say: 'To get the ball rolling, I'll need your ID number. Think of it as our secret handshake to keep things secure! 🤝'
          - For income: 'Between us, what's the monthly treasure chest looking like? (Monthly Income) 💰'
        4. GROUNDING: Every decision MUST follow the 'Credit Policy' found in the RAG documents.
        5. COMPLIANCE: If a user is denied provide empathetic adverse action notice with top 5 reasons based on credit policy and suggest ways to improve their chances for loan approval, switch to a 'Calm, Reassuring' tone. Never promise 100% approval.
        """

    # 2. Call Gemini
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=message,
        config={'system_instruction': system_instruction}
    )

    return response.text

# --- LAUNCH UI ---
# This creates a beautiful, working chat window directly in Colab
demo = gr.ChatInterface(
    fn=loan_agent_chat,
    title="Artha",
    description="Ask about your loan eligibility. Grounded in Gemini 2.5 Flash.",
    theme="soft"
)

# share=True creates a public URL that won't give you 'Failed to Fetch' errors
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8709b8a8063f4f23f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [118]:
from google.genai import types

# 1. SETUP THE TOOL (The standard 2025 Syntax)
# Ensure PROJECT_ID is 'soral-vertex-a'
# Ensure DATASTORE_ID is 'artha-connector_1766452201214_gcs_store'
DATA_STORE_PATH = f"projects/soral-vertex-a/locations/global/collections/default_collection/dataStores/artha-connector_1766452201214_gcs_store"

rag_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_ai_search=types.VertexAISearch(
            datastore=DATA_STORE_PATH
        )
    )
)

# 2. THE CHAT FUNCTION
def loan_agent_chat(user_query, history):
    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash-001",
            contents=user_query,
            config=types.GenerateContentConfig(
                tools=[rag_tool],
                # This ensures Loomis is playful but uses the PDF
                system_instruction="You are Loomis, the playful Artha underwriter. Use the credit policy PDF to decide."
            )
        )
        return response.text
    except Exception as e:
        # If there is a permission error, it will show here instead of crashing
        return f"Artha Vault Connection Error: {str(e)}"